# 09 - Enterprise Silver to Gold

Builds reconciled airline, baggage, staffing, commercial, turnaround, customer-experience, persona, and governed Data Agent products. All revenue, capacity, customer, and workforce values are synthetic proxies; recommendations are advisory and require authorized human review when consequential.

In [ ]:
from pyspark.sql import functions as F

config = spark.table('bronze_demo_config').first().asDict()
assert int(config['random_seed']) >= 0 and config['is_synthetic'] is True
observation_ts = config['observation_timestamp'].strftime('%Y-%m-%d %H:%M:%S')


def build(sql_text, table_name):
    spark.sql(sql_text.replace('__OBSERVATION_TS__', observation_ts))
    print(table_name, spark.table(table_name).count())

In [ ]:
# Explicit Gold star-schema contracts. Refresh strategy is idempotent full replacement for the configured demo window.
gold_star_sources = {
    'gold_dim_date':'dim_date','gold_dim_time':'dim_time','gold_dim_airport':'dim_airport',
    'gold_dim_terminal':'dim_terminal','gold_dim_zone':'dim_zone','gold_dim_gate':'dim_gate',
    'gold_dim_stand':'dim_stand','gold_dim_airline':'dim_airline','gold_dim_aircraft_type':'dim_aircraft',
    'gold_dim_route':'dim_route','gold_dim_employee':'dim_employee','gold_dim_team':'dim_work_team',
    'gold_dim_asset':'dim_asset','gold_dim_retail_outlet':'dim_retail_outlet',
    'gold_dim_customer_segment':'dim_customer_segment','gold_fact_flight':'fact_flight_turnaround_events',
    'gold_fact_flight_rotation':'fact_aircraft_rotation','gold_fact_turnaround':'fact_flight_turnaround_events',
    'gold_fact_turnaround_milestone':'fact_turnaround_phase','gold_fact_passenger_flow':'fact_zone_occupancy',
    'gold_fact_queue':'fact_passenger_queue_metrics','gold_fact_baggage':'fact_baggage_journey',
    'gold_fact_roster':'fact_employee_roster','gold_fact_maintenance':'fact_maintenance_work_order',
    'gold_fact_asset_inspection':'fact_asset_inspection','gold_fact_asset_state':'fact_asset_state',
    'gold_fact_energy':'fact_energy_metering','gold_fact_retail_transaction':'fact_retail_pos',
    'gold_fact_retail_inventory':'fact_retail_inventory','gold_fact_incident':'fact_operational_incidents',
    'gold_fact_customer_experience':'fact_customer_experience','gold_fact_recommendation':'fact_recommendation'}
for target_table, source_table in gold_star_sources.items():
    spark.sql(f"CREATE OR REPLACE TABLE {target_table} AS SELECT * FROM {source_table}")
    assert spark.table(target_table).count() == spark.table(source_table).count()
print('Gold star contracts', len(gold_star_sources))

In [ ]:
build("""
CREATE OR REPLACE TABLE gold_airline_route_performance AS
WITH booking AS (
  SELECT flight_event_id, COUNT(*) AS booked_passengers,
         SUM(ticket_revenue_proxy) AS ticket_revenue_proxy
  FROM fact_booking GROUP BY flight_event_id
), baggage AS (
  SELECT flight_event_id, COUNT(*) AS checked_bags,
         SUM(CASE WHEN mishandled_flag THEN 1 ELSE 0 END) AS mishandled_bags
  FROM fact_baggage_journey GROUP BY flight_event_id
), cx AS (
  SELECT flight_event_id, SUM(respondent_count) AS respondents,
         SUM(satisfaction_score * respondent_count) / SUM(respondent_count) AS satisfaction_score,
         SUM(nps_proxy * respondent_count) / SUM(respondent_count) AS nps_proxy
  FROM fact_customer_experience GROUP BY flight_event_id
)
SELECT f.airport_id, f.airline_id, br.route_id, r.destination_airport_id,
       COUNT(*) AS flights, SUM(f.passenger_count) AS passengers,
       SUM(ac.seats) AS available_seats,
       ROUND(SUM(f.passenger_count) * 100.0 / SUM(ac.seats), 1) AS load_factor_pct,
       ROUND(AVG(f.on_time_flag) * 100.0, 1) AS on_time_departure_pct,
       ROUND(AVG(f.turnaround_minutes), 1) AS avg_turnaround_min,
       ROUND(AVG(f.departure_delay_minutes), 1) AS avg_departure_delay_min,
       SUM(COALESCE(b.booked_passengers, 0)) AS booked_passengers,
       ROUND(SUM(COALESCE(b.ticket_revenue_proxy, 0.0)), 2) AS ticket_revenue_proxy,
       SUM(COALESCE(bg.checked_bags, 0)) AS checked_bags,
       SUM(COALESCE(bg.mishandled_bags, 0)) AS mishandled_bags,
       ROUND(SUM(COALESCE(bg.mishandled_bags, 0)) * 1000.0 /
             GREATEST(SUM(COALESCE(bg.checked_bags, 0)), 1), 2) AS mishandled_bags_per_1000,
       ROUND(AVG(cx.satisfaction_score), 2) AS satisfaction_score,
       ROUND(AVG(cx.nps_proxy), 1) AS nps_proxy,
       CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp,
       true AS is_synthetic
FROM fact_flight_turnaround_events f
JOIN bridge_flight_route br ON f.flight_event_id = br.flight_event_id
JOIN dim_route r ON br.route_id = r.route_id
JOIN dim_aircraft ac ON f.aircraft_type_id = ac.aircraft_type_id
LEFT JOIN booking b ON f.flight_event_id = b.flight_event_id
LEFT JOIN baggage bg ON f.flight_event_id = bg.flight_event_id
LEFT JOIN cx ON f.flight_event_id = cx.flight_event_id
GROUP BY f.airport_id, f.airline_id, br.route_id, r.destination_airport_id
""", 'gold_airline_route_performance')

build("""
CREATE OR REPLACE TABLE gold_baggage_performance AS
SELECT origin_airport_id AS airport_id, destination_airport_id,
       COUNT(*) AS checked_bags,
       SUM(CASE WHEN journey_status = 'Delivered' THEN 1 ELSE 0 END) AS delivered_bags,
       SUM(CASE WHEN mishandled_flag THEN 1 ELSE 0 END) AS mishandled_bags,
       ROUND(SUM(CASE WHEN mishandled_flag THEN 1 ELSE 0 END) * 1000.0 / COUNT(*), 2) AS mishandled_bags_per_1000,
       ROUND(AVG(journey_minutes), 1) AS avg_baggage_journey_min,
       ROUND(AVG(CASE WHEN journey_minutes <= 180 AND NOT mishandled_flag THEN 1.0 ELSE 0.0 END) * 100, 1) AS within_demo_sla_pct,
       CASE WHEN SUM(CASE WHEN mishandled_flag THEN 1 ELSE 0 END) * 1000.0 / COUNT(*) > 15 THEN 'Action'
            WHEN SUM(CASE WHEN mishandled_flag THEN 1 ELSE 0 END) * 1000.0 / COUNT(*) > 8 THEN 'Watch'
            ELSE 'Normal' END AS baggage_status,
       CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp,
       true AS is_synthetic
FROM fact_baggage_journey
GROUP BY origin_airport_id, destination_airport_id
""", 'gold_baggage_performance')

In [ ]:
build("""
CREATE OR REPLACE TABLE gold_workforce_coverage AS
SELECT r.airport_id, r.work_team_id, t.discipline, t.team_type, r.shift_name,
       COUNT(DISTINCT r.employee_id) AS rostered_employees,
       ROUND(SUM(r.planned_hours), 1) AS planned_hours,
       COUNT(DISTINCT r.assigned_gate_id) AS covered_gates,
       ROUND(LEAST(100.0, COUNT(DISTINCT r.employee_id) * 100.0 /
             GREATEST(COUNT(DISTINCT r.assigned_gate_id) * 2, 1)), 1) AS staffing_coverage_pct,
       SUM(CASE WHEN e.training_status <> 'Current' THEN 1 ELSE 0 END) AS training_exceptions,
       CASE WHEN COUNT(DISTINCT r.employee_id) < COUNT(DISTINCT r.assigned_gate_id) THEN 'Action'
            WHEN COUNT(DISTINCT r.employee_id) < COUNT(DISTINCT r.assigned_gate_id) * 2 THEN 'Watch'
            ELSE 'Covered' END AS staffing_status,
       CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp,
       true AS is_synthetic
FROM fact_employee_roster r
JOIN dim_employee e ON r.employee_id = e.employee_id
JOIN dim_work_team t ON r.work_team_id = t.work_team_id
GROUP BY r.airport_id, r.work_team_id, t.discipline, t.team_type, r.shift_name
""", 'gold_workforce_coverage')

build("""
CREATE OR REPLACE TABLE gold_retail_performance AS
WITH outlet AS (
  SELECT outlet_id, airport_id, terminal_id,
         SUM(transaction_count) AS transactions,
         SUM(gross_sales_proxy) AS gross_sales_proxy,
         SUM(refund_proxy) AS refund_proxy,
         SUM(gross_sales_proxy - refund_proxy) AS net_revenue_proxy,
         SUM(gross_sales_proxy) / GREATEST(SUM(transaction_count), 1) AS average_basket_proxy
  FROM fact_retail_pos
  GROUP BY outlet_id, airport_id, terminal_id
), pax AS (
  SELECT airport_id, SUM(passenger_count) AS departing_passengers
  FROM fact_flight_turnaround_events GROUP BY airport_id
)
SELECT o.airport_id, o.terminal_id, o.outlet_id, d.outlet_category,
       o.transactions, ROUND(o.gross_sales_proxy, 2) AS gross_sales_proxy,
       ROUND(o.refund_proxy, 2) AS refund_proxy,
       ROUND(o.net_revenue_proxy, 2) AS net_revenue_proxy,
       ROUND(o.average_basket_proxy, 2) AS average_basket_proxy,
       p.departing_passengers,
       ROUND(o.net_revenue_proxy / GREATEST(p.departing_passengers, 1), 2) AS revenue_per_departing_passenger_proxy,
       CASE WHEN o.refund_proxy / GREATEST(o.gross_sales_proxy, 1) > 0.04 THEN 'Watch' ELSE 'Normal' END AS commercial_status,
       CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp,
       true AS is_synthetic
FROM outlet o
JOIN dim_retail_outlet d ON o.outlet_id = d.outlet_id
JOIN pax p ON o.airport_id = p.airport_id
""", 'gold_retail_performance')

In [ ]:
build("""
CREATE OR REPLACE TABLE gold_customer_experience AS
SELECT airport_id, route_id, customer_segment,
       SUM(respondent_count) AS respondents,
       ROUND(SUM(satisfaction_score * respondent_count) / SUM(respondent_count), 2) AS satisfaction_score,
       ROUND(SUM(nps_proxy * respondent_count) / SUM(respondent_count), 1) AS nps_proxy,
       CASE WHEN SUM(satisfaction_score * respondent_count) / SUM(respondent_count) < 3.5 THEN 'Action'
            WHEN SUM(satisfaction_score * respondent_count) / SUM(respondent_count) < 4.0 THEN 'Watch'
            ELSE 'Positive' END AS customer_experience_status,
       CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp,
       true AS is_synthetic
FROM fact_customer_experience
GROUP BY airport_id, route_id, customer_segment
""", 'gold_customer_experience')

build("""
CREATE OR REPLACE TABLE gold_turnaround_phase_performance AS
SELECT airport_id, gate_id, phase_name, phase_sequence,
       COUNT(*) AS observed_flights,
       ROUND(AVG(phase_duration_min), 2) AS avg_phase_duration_min,
       ROUND(MAX(phase_duration_min), 2) AS max_phase_duration_min,
       SUM(CASE WHEN milestone_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed_milestones,
       ROUND(AVG(CASE WHEN milestone_status = 'OnPlan' THEN 1.0 ELSE 0.0 END) * 100, 1) AS milestone_adherence_pct,
       CASE WHEN AVG(CASE WHEN milestone_status = 'OnPlan' THEN 1.0 ELSE 0.0 END) < 0.8 THEN 'Bottleneck'
            WHEN AVG(CASE WHEN milestone_status = 'OnPlan' THEN 1.0 ELSE 0.0 END) < 0.95 THEN 'Watch'
            ELSE 'OnPlan' END AS phase_status,
       CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp,
       true AS is_synthetic
FROM fact_turnaround_phase
GROUP BY airport_id, gate_id, phase_name, phase_sequence
""", 'gold_turnaround_phase_performance')

In [ ]:
# Domain KPI products. Every target/benchmark/risk value is a documented synthetic assumption.
build("""
CREATE OR REPLACE TABLE gold_flight_operations_kpi AS
WITH conflict AS (
  SELECT a.airport_id, COUNT(*) AS gate_conflict_count
  FROM fact_flight_turnaround_events a JOIN fact_flight_turnaround_events b
    ON a.gate_id=b.gate_id AND a.flight_event_id < b.flight_event_id
   AND a.actual_arrival < b.actual_departure AND b.actual_arrival < a.actual_departure
  GROUP BY a.airport_id),
base AS (
  SELECT f.airport_id, COUNT(*) AS flights,
    ROUND(AVG(f.on_time_arrival_flag)*100,1) AS on_time_arrival_pct,
    ROUND(AVG(f.on_time_flag)*100,1) AS on_time_departure_pct,
    ROUND(AVG(f.arrival_delay_minutes),1) AS avg_arrival_delay_min,
    ROUND(AVG(f.departure_delay_minutes),1) AS avg_departure_delay_min,
    ROUND(AVG(f.turnaround_minutes),1) AS avg_turnaround_min,
    ROUND(AVG(CASE WHEN f.turnaround_minutes <= a.turnaround_target_min THEN 1.0 ELSE 0.0 END)*100,1) AS turnaround_target_attainment_pct,
    ROUND(AVG(CASE WHEN f.arrival_delay_minutes > 15 THEN 1.0 ELSE 0.0 END)*100,1) AS late_inbound_contribution_pct
  FROM fact_flight_turnaround_events f JOIN dim_aircraft a ON f.aircraft_type_id=a.aircraft_type_id
  GROUP BY f.airport_id),
milestone AS (SELECT airport_id, ROUND(AVG(CASE WHEN milestone_status='OnPlan' THEN 1.0 ELSE 0.0 END)*100,1) AS milestone_adherence_pct FROM fact_turnaround_phase GROUP BY airport_id),
utilization AS (SELECT airport_id, ROUND(AVG(utilization_pct),1) AS gate_utilization_pct, ROUND(AVG(utilization_pct),1) AS stand_utilization_pct FROM gold_gate_utilization GROUP BY airport_id)
SELECT b.*, m.milestone_adherence_pct, u.gate_utilization_pct, u.stand_utilization_pct,
       COALESCE(c.gate_conflict_count,0) AS gate_conflict_count,
       38.0 AS synthetic_narrow_body_target_min,
       CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp,
       'DerivedAnalytical' AS data_classification, true AS is_synthetic
FROM base b JOIN milestone m ON b.airport_id=m.airport_id JOIN utilization u ON b.airport_id=u.airport_id
LEFT JOIN conflict c ON b.airport_id=c.airport_id
""", 'gold_flight_operations_kpi')

build("""
CREATE OR REPLACE TABLE gold_passenger_flow_kpi AS
WITH queue AS (
 SELECT airport_id, ROUND(AVG(queue_length),1) AS avg_queue_length,
   MAX(queue_length) AS peak_queue_length, ROUND(AVG(wait_time_min),1) AS avg_wait_min,
   MAX(wait_time_min) AS peak_wait_min, SUM(throughput_pax) AS throughput_pax,
   ROUND(AVG(CASE WHEN wait_time_min >= 15 THEN 1.0 ELSE 0.0 END)*100,1) AS predicted_congestion_risk_pct
 FROM fact_passenger_queue_metrics GROUP BY airport_id),
boarding AS (
 SELECT f.airport_id, ROUND(AVG(CASE WHEN b.boarding_window_risk='Watch' THEN 1.0 ELSE 0.0 END)*100,1) AS boarding_window_risk_pct
 FROM fact_boarding_event b JOIN fact_flight_turnaround_events f ON b.flight_event_id=f.flight_event_id GROUP BY f.airport_id),
connection AS (
 SELECT airport_id, ROUND(AVG(CASE WHEN arrival_delay_minutes > 30 THEN 1.0 ELSE 0.0 END)*100,1) AS missed_connection_risk_pct
 FROM fact_flight_turnaround_events GROUP BY airport_id)
SELECT q.*, b.boarding_window_risk_pct, c.missed_connection_risk_pct,
       15.0 AS synthetic_wait_watch_threshold_min, 45.0 AS synthetic_connection_window_min,
       CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp,
       'DerivedAnalytical' AS data_classification, true AS is_synthetic
FROM queue q JOIN boarding b ON q.airport_id=b.airport_id JOIN connection c ON q.airport_id=c.airport_id
""", 'gold_passenger_flow_kpi')

build("""
CREATE OR REPLACE TABLE gold_baggage_kpi AS
WITH scans AS (SELECT bag_token, COUNT(*) AS actual_scans FROM fact_baggage_scan GROUP BY bag_token),
base AS (
 SELECT b.origin_airport_id AS airport_id, COUNT(*) AS bags_processed,
   SUM(CASE WHEN b.mishandled_flag THEN 1 ELSE 0 END) AS mishandled_bags,
   ROUND(AVG(b.journey_minutes),1) AS avg_delivery_time_min,
   ROUND(AVG(CASE WHEN b.journey_minutes > 120 OR b.mishandled_flag THEN 1.0 ELSE 0.0 END)*100,1) AS transfer_bag_risk_pct,
   ROUND(AVG(LEAST(s.actual_scans*100.0/b.expected_scan_count,100.0)),1) AS scan_completeness_pct
 FROM fact_baggage_journey b JOIN scans s ON b.bag_token=s.bag_token GROUP BY b.origin_airport_id),
flights AS (SELECT airport_id, COUNT(*) AS flights FROM fact_flight_turnaround_events GROUP BY airport_id)
SELECT b.airport_id, b.bags_processed, ROUND(b.bags_processed*1.0/f.flights,1) AS bags_per_flight,
       ROUND(b.mishandled_bags*1000.0/b.bags_processed,2) AS mishandled_bags_per_1000,
       b.transfer_bag_risk_pct, b.avg_delivery_time_min, b.scan_completeness_pct,
       CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp,
       'DerivedAnalytical' AS data_classification, true AS is_synthetic
FROM base b JOIN flights f ON b.airport_id=f.airport_id
""", 'gold_baggage_kpi')

In [ ]:
build("""
CREATE OR REPLACE TABLE gold_workforce_kpi AS
WITH roster AS (
 SELECT airport_id, COUNT(DISTINCT employee_id) AS rostered_employees,
   ROUND(SUM(planned_hours),1) AS planned_hours, ROUND(SUM(actual_hours),1) AS actual_hours,
   ROUND(SUM(overtime_hours),1) AS overtime_hours,
   ROUND(AVG(CASE WHEN actual_hours >= planned_hours*0.9 THEN 1.0 ELSE 0.0 END)*100,1) AS roster_coverage_pct
 FROM fact_employee_roster GROUP BY airport_id),
skill AS (
 SELECT e.home_airport_id AS airport_id, ROUND(COUNT(DISTINCT s.employee_id)*100.0/COUNT(DISTINCT e.employee_id),1) AS skill_coverage_pct
 FROM dim_employee e LEFT JOIN bridge_employee_skill s ON e.employee_id=s.employee_id GROUP BY e.home_airport_id),
tasks AS (SELECT airport_id, COUNT(*) AS ramp_tasks FROM fact_ramp_service_task GROUP BY airport_id),
teams AS (SELECT airport_id, COUNT(*) AS teams FROM dim_work_team GROUP BY airport_id)
SELECT r.*, s.skill_coverage_pct, t.ramp_tasks, ROUND(t.ramp_tasks*1.0/g.teams,1) AS tasks_per_team,
       ROUND(r.actual_hours-r.planned_hours,1) AS staffing_recommendation_variance_hours,
       CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp,
       'DerivedAnalytical' AS data_classification, true AS is_synthetic
FROM roster r JOIN skill s ON r.airport_id=s.airport_id JOIN tasks t ON r.airport_id=t.airport_id JOIN teams g ON r.airport_id=g.airport_id
""", 'gold_workforce_kpi')

build("""
CREATE OR REPLACE TABLE gold_maintenance_kpi AS
WITH asset AS (
 SELECT airport_id, ROUND(AVG(availability_pct),1) AS asset_availability_pct,
   SUM(CASE WHEN anomaly_flag THEN 1 ELSE 0 END) AS failure_count,
   SUM(CASE WHEN anomaly_flag THEN 1 ELSE 0 END) AS anomaly_count,
   COUNT(*) AS asset_observations
 FROM fact_asset_state GROUP BY airport_id),
work AS (
 SELECT airport_id, ROUND(AVG(resolution_hours),1) AS mean_time_to_repair_hours,
   ROUND(AVG(CASE WHEN work_order_type='Preventive' AND status='Closed' THEN 1.0 WHEN work_order_type='Preventive' THEN 0.0 END)*100,1) AS preventive_maintenance_compliance_pct,
   SUM(CASE WHEN status<>'Closed' THEN 1 ELSE 0 END) AS open_work_order_backlog
 FROM fact_maintenance_work_order GROUP BY airport_id),
inspection AS (
 SELECT airport_id, ROUND(AVG(inspection_score),1) AS avg_inspection_score,
   ROUND(AVG(CASE WHEN inspection_status='Pass' THEN 1.0 ELSE 0.0 END)*100,1) AS inspection_compliance_pct,
   SUM(CASE WHEN follow_up_required THEN 1 ELSE 0 END) AS inspection_follow_up_count
 FROM fact_asset_inspection GROUP BY airport_id)
SELECT a.airport_id, a.asset_availability_pct, a.failure_count,
       ROUND((24.0*a.asset_observations)/GREATEST(a.failure_count,1),1) AS mean_time_between_failures_hours,
       w.mean_time_to_repair_hours, COALESCE(w.preventive_maintenance_compliance_pct,100.0) AS preventive_maintenance_compliance_pct,
       w.open_work_order_backlog, a.anomaly_count,
       ROUND(a.failure_count*100.0/a.asset_observations,2) AS predicted_failure_risk_pct,
       i.avg_inspection_score, i.inspection_compliance_pct, i.inspection_follow_up_count,
       CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp,
       'DerivedAnalytical' AS data_classification, true AS is_synthetic
FROM asset a JOIN work w ON a.airport_id=w.airport_id JOIN inspection i ON a.airport_id=i.airport_id
""", 'gold_maintenance_kpi')

build("""
CREATE OR REPLACE TABLE gold_energy_sustainability_kpi AS
WITH energy AS (
 SELECT airport_id, SUM(kwh) AS total_kwh, MAX(kwh) AS peak_demand_kwh
 FROM fact_energy_metering GROUP BY airport_id),
flight AS (SELECT airport_id, COUNT(*) AS flights, SUM(passenger_count) AS passengers FROM fact_flight_turnaround_events GROUP BY airport_id)
SELECT e.airport_id, ROUND(e.total_kwh,1) AS total_kwh,
       ROUND(e.total_kwh/f.passengers,3) AS kwh_per_passenger,
       ROUND(e.total_kwh/f.flights,1) AS kwh_per_flight,
       ROUND(e.peak_demand_kwh,1) AS peak_demand_kwh,
       450.0 AS synthetic_benchmark_kwh_per_flight,
       ROUND((e.total_kwh/f.flights-450.0)*100.0/450.0,1) AS energy_benchmark_variance_pct,
       ROUND(e.total_kwh*0.233,1) AS estimated_synthetic_emissions_kg_co2e,
       CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp,
       'DerivedAnalytical' AS data_classification, true AS is_synthetic
FROM energy e JOIN flight f ON e.airport_id=f.airport_id
""", 'gold_energy_sustainability_kpi')

In [ ]:
build("""
CREATE OR REPLACE TABLE gold_commercial_kpi AS
WITH retail AS (
 SELECT airport_id, SUM(transaction_count) AS transactions,
   SUM(gross_sales_proxy-refund_proxy) AS revenue_proxy,
   SUM(gross_sales_proxy)/GREATEST(SUM(transaction_count),1) AS avg_transaction_value_proxy,
   COUNT(DISTINCT outlet_id) AS outlets
 FROM fact_retail_pos GROUP BY airport_id),
flight AS (SELECT airport_id, SUM(passenger_count) AS passengers FROM fact_flight_turnaround_events GROUP BY airport_id)
SELECT r.airport_id, ROUND(r.revenue_proxy,2) AS revenue_proxy,
       ROUND(r.revenue_proxy/f.passengers,2) AS revenue_per_passenger_proxy,
       ROUND(r.transactions*1.0/f.passengers,3) AS transactions_per_passenger,
       ROUND(LEAST(r.transactions*100.0/f.passengers,100.0),1) AS conversion_rate_pct,
       ROUND(r.avg_transaction_value_proxy,2) AS average_transaction_value_proxy,
       ROUND(r.revenue_proxy/r.outlets,2) AS outlet_performance_proxy,
       8.50 AS synthetic_benchmark_revenue_per_passenger,
       ROUND((r.revenue_proxy/f.passengers-8.50)*100.0/8.50,1) AS concession_benchmark_variance_pct,
       CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp,
       'DerivedAnalytical' AS data_classification, true AS is_synthetic
FROM retail r JOIN flight f ON r.airport_id=f.airport_id
""", 'gold_commercial_kpi')

build("""
CREATE OR REPLACE TABLE gold_incident_customer_kpi AS
WITH incident AS (
 SELECT airport_id, COUNT(*) AS incident_count,
   SUM(CASE WHEN severity='High' THEN 1 ELSE 0 END) AS high_severity_incidents,
   SUM(CASE WHEN severity='Medium' THEN 1 ELSE 0 END) AS medium_severity_incidents,
   SUM(CASE WHEN severity='Low' THEN 1 ELSE 0 END) AS low_severity_incidents,
   ROUND(AVG(CASE WHEN status='Resolved' THEN GREATEST(delay_minutes,1) END),1) AS time_to_resolution_proxy_min
 FROM fact_operational_incidents GROUP BY airport_id),
flight AS (SELECT airport_id, COUNT(*) AS flights FROM fact_flight_turnaround_events GROUP BY airport_id),
cx AS (
 SELECT airport_id, ROUND(SUM(satisfaction_score*respondent_count)/SUM(respondent_count),2) AS customer_satisfaction,
   ROUND(SUM(nps_proxy*respondent_count)/SUM(respondent_count),1) AS synthetic_nps,
   ROUND(SUM(CASE WHEN satisfaction_score<3 THEN respondent_count ELSE 0 END)*1000.0/SUM(respondent_count),2) AS complaints_per_1000_responses
 FROM fact_customer_experience GROUP BY airport_id),
recommendation AS (
 SELECT airport_id, ROUND(AVG(CASE WHEN recommendation_status='AcceptedForScenario' THEN 1.0 ELSE 0.0 END)*100,1) AS recommendation_acceptance_pct
 FROM fact_recommendation GROUP BY airport_id)
SELECT i.airport_id, i.incident_count, ROUND(i.incident_count*100.0/f.flights,2) AS incidents_per_100_flights,
       i.high_severity_incidents, i.medium_severity_incidents, i.low_severity_incidents,
       i.time_to_resolution_proxy_min, c.customer_satisfaction, c.synthetic_nps,
       c.complaints_per_1000_responses, r.recommendation_acceptance_pct,
       CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp,
       'DerivedAnalytical' AS data_classification, true AS is_synthetic
FROM incident i JOIN flight f ON i.airport_id=f.airport_id
JOIN cx c ON i.airport_id=c.airport_id JOIN recommendation r ON i.airport_id=r.airport_id
""", 'gold_incident_customer_kpi')

kpi_catalog_rows = [
    ('On-Time Arrival %','flight/airport','Flights arriving <=15 minutes late','percent','90','Actual','Synthetic target'),
    ('On-Time Departure %','flight/airport','Flights departing <=15 minutes late','percent','90','Actual','Synthetic target'),
    ('Narrow-body Turnaround','flight/airport','Average narrow-body turnaround minutes','minutes','38','Actual','Synthetic target'),
    ('Predicted Congestion Risk','queue/airport','Share of observations with wait >=15 minutes','percent','15','Forecast','Synthetic threshold'),
    ('Mishandled Bags per 1,000','bag/airport','Mishandled bags / processed bags * 1000','rate','8','Actual','Synthetic benchmark'),
    ('Staffing Coverage %','roster/airport','Actual covered roster / planned roster','percent','95','Actual','Synthetic target'),
    ('Predicted Failure Risk','asset observation/airport','Anomalous observations / all observations','percent','5','Forecast','Synthetic threshold'),
    ('Energy per Flight','meter/airport','Total kWh / flights','kWh/flight','450','Actual','Synthetic benchmark'),
    ('Revenue per Passenger','POS/airport','Net synthetic revenue / passengers','proxy units/passenger','8.5','Actual','Synthetic benchmark'),
    ('Recommendation Acceptance %','recommendation/airport','Accepted scenario recommendations / recommendations','percent','70','Actual','Synthetic target')]
kpi_catalog_schema = 'kpi_name string, grain string, formula string, unit string, target_value string, value_type string, caveat string'
kpi_catalog = spark.createDataFrame(kpi_catalog_rows, kpi_catalog_schema).withColumn('is_synthetic',F.lit(True)).withColumn('data_classification',F.lit('DerivedAnalytical'))
kpi_catalog.write.mode('overwrite').option('overwriteSchema','true').format('delta').saveAsTable('gold_kpi_catalog')

In [ ]:
# Preserve one row per configured airport when maintenance or incident observations are absent.
build("""
CREATE OR REPLACE TABLE gold_maintenance_kpi AS
WITH asset AS (
 SELECT airport_id, ROUND(AVG(availability_pct),1) AS asset_availability_pct,
   SUM(CASE WHEN anomaly_flag THEN 1 ELSE 0 END) AS failure_count,
   SUM(CASE WHEN anomaly_flag THEN 1 ELSE 0 END) AS anomaly_count,
   COUNT(*) AS asset_observations
 FROM fact_asset_state GROUP BY airport_id),
work AS (
 SELECT airport_id, ROUND(AVG(resolution_hours),1) AS mean_time_to_repair_hours,
   ROUND(AVG(CASE WHEN work_order_type='Preventive' AND status='Closed' THEN 1.0 WHEN work_order_type='Preventive' THEN 0.0 END)*100,1) AS preventive_maintenance_compliance_pct,
   SUM(CASE WHEN status<>'Closed' THEN 1 ELSE 0 END) AS open_work_order_backlog
 FROM fact_maintenance_work_order GROUP BY airport_id),
inspection AS (
 SELECT airport_id, ROUND(AVG(inspection_score),1) AS avg_inspection_score,
   ROUND(AVG(CASE WHEN inspection_status='Pass' THEN 1.0 ELSE 0.0 END)*100,1) AS inspection_compliance_pct,
   SUM(CASE WHEN follow_up_required THEN 1 ELSE 0 END) AS inspection_follow_up_count
 FROM fact_asset_inspection GROUP BY airport_id)
SELECT d.airport_id,
       COALESCE(a.asset_availability_pct,0.0) AS asset_availability_pct,
       COALESCE(a.failure_count,0) AS failure_count,
       ROUND((24.0*COALESCE(a.asset_observations,0))/GREATEST(COALESCE(a.failure_count,0),1),1) AS mean_time_between_failures_hours,
       COALESCE(w.mean_time_to_repair_hours,0.0) AS mean_time_to_repair_hours,
       COALESCE(w.preventive_maintenance_compliance_pct,100.0) AS preventive_maintenance_compliance_pct,
       COALESCE(w.open_work_order_backlog,0) AS open_work_order_backlog,
       COALESCE(a.anomaly_count,0) AS anomaly_count,
       ROUND(COALESCE(a.failure_count,0)*100.0/GREATEST(COALESCE(a.asset_observations,0),1),2) AS predicted_failure_risk_pct,
       COALESCE(i.avg_inspection_score,0.0) AS avg_inspection_score,
       COALESCE(i.inspection_compliance_pct,0.0) AS inspection_compliance_pct,
       COALESCE(i.inspection_follow_up_count,0) AS inspection_follow_up_count,
       CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp,
       'DerivedAnalytical' AS data_classification, true AS is_synthetic
FROM gold_dim_airport d
LEFT JOIN asset a ON d.airport_id=a.airport_id
LEFT JOIN work w ON d.airport_id=w.airport_id
LEFT JOIN inspection i ON d.airport_id=i.airport_id
""", 'gold_maintenance_kpi')

build("""
CREATE OR REPLACE TABLE gold_incident_customer_kpi AS
WITH incident AS (
 SELECT airport_id, COUNT(*) AS incident_count,
   SUM(CASE WHEN severity='High' THEN 1 ELSE 0 END) AS high_severity_incidents,
   SUM(CASE WHEN severity='Medium' THEN 1 ELSE 0 END) AS medium_severity_incidents,
   SUM(CASE WHEN severity='Low' THEN 1 ELSE 0 END) AS low_severity_incidents,
   ROUND(AVG(CASE WHEN status='Resolved' THEN GREATEST(delay_minutes,1) END),1) AS time_to_resolution_proxy_min
 FROM fact_operational_incidents GROUP BY airport_id),
flight AS (SELECT airport_id, COUNT(*) AS flights FROM fact_flight_turnaround_events GROUP BY airport_id),
cx AS (
 SELECT airport_id, ROUND(SUM(satisfaction_score*respondent_count)/SUM(respondent_count),2) AS customer_satisfaction,
   ROUND(SUM(nps_proxy*respondent_count)/SUM(respondent_count),1) AS synthetic_nps,
   ROUND(SUM(CASE WHEN satisfaction_score<3 THEN respondent_count ELSE 0 END)*1000.0/SUM(respondent_count),2) AS complaints_per_1000_responses
 FROM fact_customer_experience GROUP BY airport_id),
recommendation AS (
 SELECT airport_id, ROUND(AVG(CASE WHEN recommendation_status='AcceptedForScenario' THEN 1.0 ELSE 0.0 END)*100,1) AS recommendation_acceptance_pct
 FROM fact_recommendation GROUP BY airport_id)
SELECT d.airport_id,
       COALESCE(i.incident_count,0) AS incident_count,
       ROUND(COALESCE(i.incident_count,0)*100.0/GREATEST(COALESCE(f.flights,0),1),2) AS incidents_per_100_flights,
       COALESCE(i.high_severity_incidents,0) AS high_severity_incidents,
       COALESCE(i.medium_severity_incidents,0) AS medium_severity_incidents,
       COALESCE(i.low_severity_incidents,0) AS low_severity_incidents,
       COALESCE(i.time_to_resolution_proxy_min,0.0) AS time_to_resolution_proxy_min,
       COALESCE(c.customer_satisfaction,0.0) AS customer_satisfaction,
       COALESCE(c.synthetic_nps,0.0) AS synthetic_nps,
       COALESCE(c.complaints_per_1000_responses,0.0) AS complaints_per_1000_responses,
       COALESCE(r.recommendation_acceptance_pct,0.0) AS recommendation_acceptance_pct,
       CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp,
       'DerivedAnalytical' AS data_classification, true AS is_synthetic
FROM gold_dim_airport d
LEFT JOIN incident i ON d.airport_id=i.airport_id
LEFT JOIN flight f ON d.airport_id=f.airport_id
LEFT JOIN cx c ON d.airport_id=c.airport_id
LEFT JOIN recommendation r ON d.airport_id=r.airport_id
""", 'gold_incident_customer_kpi')

In [ ]:
build("""
CREATE OR REPLACE TABLE gold_aircraft_rotation_kpi AS
WITH rotation AS (
 SELECT f.airport_id, COUNT(*) AS rotation_legs, COUNT(DISTINCT r.aircraft_instance_id) AS aircraft_instances,
   ROUND(AVG(COALESCE(r.ground_interval_min,0.0)),1) AS avg_ground_interval_min,
   SUM(CASE WHEN r.overlap_flag THEN 1 ELSE 0 END) AS overlap_count
 FROM fact_aircraft_rotation r JOIN fact_flight_turnaround_events f ON r.flight_event_id=f.flight_event_id
 GROUP BY f.airport_id)
SELECT *, ROUND(rotation_legs*1.0/GREATEST(aircraft_instances,1),1) AS legs_per_aircraft,
       CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp,
       'DerivedAnalytical' AS data_classification, true AS is_synthetic
FROM rotation
""", 'gold_aircraft_rotation_kpi')

build("""
CREATE OR REPLACE TABLE gold_retail_inventory_kpi AS
SELECT airport_id, COUNT(*) AS inventory_snapshots, SUM(on_hand_units) AS total_on_hand_units,
       SUM(CASE WHEN stock_status='Reorder' THEN 1 ELSE 0 END) AS reorder_items,
       ROUND(AVG(CASE WHEN stock_status='Reorder' THEN 1.0 ELSE 0.0 END)*100,1) AS reorder_rate_pct,
       ROUND(AVG(on_hand_units),1) AS avg_on_hand_units,
       CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp,
       'DerivedAnalytical' AS data_classification, true AS is_synthetic
FROM fact_retail_inventory GROUP BY airport_id
""", 'gold_retail_inventory_kpi')

In [ ]:
build("""
CREATE OR REPLACE TABLE gold_persona_scorecard AS
WITH airline AS (
  SELECT airport_id, ROUND(AVG(on_time_departure_pct), 1) AS otp, ROUND(AVG(load_factor_pct), 1) AS load_factor
  FROM gold_airline_route_performance GROUP BY airport_id
), baggage AS (
  SELECT airport_id, ROUND(AVG(within_demo_sla_pct), 1) AS baggage_sla, ROUND(AVG(mishandled_bags_per_1000), 2) AS mishandled_rate
  FROM gold_baggage_performance GROUP BY airport_id
), staffing AS (
  SELECT airport_id, ROUND(AVG(staffing_coverage_pct), 1) AS staffing_coverage
  FROM gold_workforce_coverage GROUP BY airport_id
), commercial AS (
  SELECT airport_id, ROUND(SUM(net_revenue_proxy), 2) AS net_revenue, ROUND(AVG(average_basket_proxy), 2) AS avg_basket
  FROM gold_retail_performance GROUP BY airport_id
), cx AS (
  SELECT airport_id, ROUND(AVG(satisfaction_score), 2) AS satisfaction, ROUND(AVG(nps_proxy), 1) AS nps
  FROM gold_customer_experience GROUP BY airport_id
), maintenance AS (
  SELECT airport_id, ROUND(AVG(availability_pct), 1) AS availability,
         SUM(anomaly_count + maintenance_anomaly_count) AS anomalies
  FROM gold_asset_reliability GROUP BY airport_id
), it AS (
  SELECT ROUND(AVG(data_quality_pass_pct), 1) AS data_quality,
         ROUND(AVG(synthetic_capacity_usage_pct), 1) AS capacity_proxy
  FROM gold_it_service_health
), base AS (
  SELECT d.airport_id,
         COALESCE(k.on_time_departure_rate, 0.0) AS on_time_departure_rate,
         COALESCE(k.avg_turnaround_min, 0.0) AS avg_turnaround_min,
         COALESCE(k.avg_queue_wait_min, 0.0) AS avg_queue_wait_min,
         COALESCE(h.operational_risk_score, 0.0) AS operational_risk_score,
         COALESCE(a.otp, 0.0) AS otp, COALESCE(a.load_factor, 0.0) AS load_factor,
         COALESCE(b.baggage_sla, 0.0) AS baggage_sla, COALESCE(b.mishandled_rate, 0.0) AS mishandled_rate,
         COALESCE(s.staffing_coverage, 0.0) AS staffing_coverage,
         COALESCE(c.net_revenue, 0.0) AS net_revenue, COALESCE(c.avg_basket, 0.0) AS avg_basket,
         COALESCE(x.satisfaction, 0.0) AS satisfaction, COALESCE(x.nps, 0.0) AS nps,
         COALESCE(m.availability, 0.0) AS availability, COALESCE(m.anomalies, 0) AS anomalies,
         COALESCE(i.data_quality, 0.0) AS data_quality, COALESCE(i.capacity_proxy, 0.0) AS capacity_proxy
  FROM gold_dim_airport d
  LEFT JOIN gold_kpi_daily_summary k ON d.airport_id = k.airport_id
  LEFT JOIN gold_airport_operational_health h ON d.airport_id = h.airport_id
  LEFT JOIN airline a ON d.airport_id = a.airport_id
  LEFT JOIN baggage b ON d.airport_id = b.airport_id
  LEFT JOIN staffing s ON d.airport_id = s.airport_id
  LEFT JOIN commercial c ON d.airport_id = c.airport_id
  LEFT JOIN cx x ON d.airport_id = x.airport_id
  LEFT JOIN maintenance m ON d.airport_id = m.airport_id
  CROSS JOIN it i
)
SELECT airport_id, persona, primary_kpi_name, ROUND(primary_kpi_value, 2) AS primary_kpi_value,
       secondary_kpi_name, ROUND(secondary_kpi_value, 2) AS secondary_kpi_value,
       scorecard_status, CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp,
       true AS advisory_only, true AS is_synthetic
FROM base
LATERAL VIEW STACK(7,
  'Airport', 'On-time departure pct', CAST(on_time_departure_rate AS DOUBLE), 'Queue wait min', CAST(avg_queue_wait_min AS DOUBLE), CASE WHEN on_time_departure_rate < 80 THEN 'Action' WHEN on_time_departure_rate < 90 THEN 'Watch' ELSE 'Normal' END,
  'Airline', 'Airline on-time pct', CAST(otp AS DOUBLE), 'Load factor pct', CAST(load_factor AS DOUBLE), CASE WHEN otp < 80 THEN 'Action' WHEN otp < 90 THEN 'Watch' ELSE 'Normal' END,
  'Executive', 'Operational risk score', CAST(operational_risk_score AS DOUBLE), 'Synthetic net revenue', CAST(net_revenue AS DOUBLE), CASE WHEN operational_risk_score >= 60 THEN 'Action' WHEN operational_risk_score >= 35 THEN 'Watch' ELSE 'Normal' END,
  'Operations', 'Turnaround min', CAST(avg_turnaround_min AS DOUBLE), 'Staffing coverage pct', CAST(staffing_coverage AS DOUBLE), CASE WHEN staffing_coverage < 80 THEN 'Action' WHEN avg_turnaround_min > 60 THEN 'Watch' ELSE 'Normal' END,
  'Maintenance', 'Asset availability pct', CAST(availability AS DOUBLE), 'Anomaly observations', CAST(anomalies AS DOUBLE), CASE WHEN availability < 95 OR anomalies > 10 THEN 'Action' WHEN availability < 98 OR anomalies > 0 THEN 'Watch' ELSE 'Normal' END,
  'Commercial', 'Synthetic net revenue', CAST(net_revenue AS DOUBLE), 'Customer satisfaction', CAST(satisfaction AS DOUBLE), CASE WHEN satisfaction < 3.5 THEN 'Action' WHEN satisfaction < 4.0 THEN 'Watch' ELSE 'Normal' END,
  'IT', 'Data quality pass pct', CAST(data_quality AS DOUBLE), 'Synthetic capacity pct', CAST(capacity_proxy AS DOUBLE), CASE WHEN data_quality < 100 THEN 'Action' WHEN capacity_proxy > 80 THEN 'Watch' ELSE 'Normal' END
) scorecards AS persona, primary_kpi_name, primary_kpi_value, secondary_kpi_name, secondary_kpi_value, scorecard_status
""", 'gold_persona_scorecard')

In [ ]:
# Preserve the most severe status when an airport has mixed route or shift states.
build("""
CREATE OR REPLACE TABLE gold_data_agent_enterprise_context AS
WITH baggage AS (
  SELECT airport_id, ROUND(AVG(mishandled_bags_per_1000), 2) AS mishandled_bags_per_1000,
         MAX(CASE baggage_status WHEN 'Action' THEN 3 WHEN 'Watch' THEN 2 ELSE 1 END) AS baggage_status_rank
  FROM gold_baggage_performance GROUP BY airport_id
), staffing AS (
  SELECT airport_id, ROUND(AVG(staffing_coverage_pct), 1) AS staffing_coverage_pct,
         MAX(CASE staffing_status WHEN 'Action' THEN 3 WHEN 'Watch' THEN 2 ELSE 1 END) AS staffing_status_rank
  FROM gold_workforce_coverage GROUP BY airport_id
), retail AS (
  SELECT airport_id, ROUND(SUM(net_revenue_proxy), 2) AS net_revenue_proxy
  FROM gold_retail_performance GROUP BY airport_id
), cx AS (
  SELECT airport_id, ROUND(AVG(satisfaction_score), 2) AS satisfaction_score,
         ROUND(AVG(nps_proxy), 1) AS nps_proxy
  FROM gold_customer_experience GROUP BY airport_id
)
SELECT d.airport_id,
       COALESCE(h.risk_category, 'NoCurrentSignal') AS risk_category,
       COALESCE(h.operational_risk_score, 0.0) AS operational_risk_score,
       COALESCE(b.mishandled_bags_per_1000, 0.0) AS mishandled_bags_per_1000,
       CASE COALESCE(b.baggage_status_rank, 1) WHEN 3 THEN 'Action' WHEN 2 THEN 'Watch' ELSE 'Normal' END AS baggage_status,
       COALESCE(s.staffing_coverage_pct, 0.0) AS staffing_coverage_pct,
       CASE COALESCE(s.staffing_status_rank, 1) WHEN 3 THEN 'Action' WHEN 2 THEN 'Watch' ELSE 'Covered' END AS staffing_status,
       COALESCE(r.net_revenue_proxy, 0.0) AS net_revenue_proxy,
       COALESCE(x.satisfaction_score, 0.0) AS satisfaction_score,
       COALESCE(x.nps_proxy, 0.0) AS nps_proxy,
       CASE WHEN b.baggage_status_rank = 3 THEN 'Authorized baggage operations lead should review synthetic exception patterns; no operational action is issued'
            WHEN s.staffing_status_rank = 3 THEN 'Authorized workforce lead should review synthetic roster coverage; no staff assignment is issued'
            WHEN x.airport_id IS NULL THEN 'Authorized customer-experience lead should review missing current synthetic evidence'
            WHEN x.satisfaction_score < 3.5 THEN 'Authorized customer-experience lead should review synthetic service-recovery options'
            ELSE 'Continue governed monitoring; no consequential action is recommended' END AS recommendation_text,
       CASE WHEN b.baggage_status_rank = 3 OR s.staffing_status_rank = 3 OR x.airport_id IS NULL OR x.satisfaction_score < 3.5 THEN true ELSE false END AS human_approval_required,
       'gold_airport_operational_health;gold_baggage_performance;gold_workforce_coverage;gold_retail_performance;gold_customer_experience' AS source_table_references,
       CAST('__OBSERVATION_TS__' AS TIMESTAMP) AS observation_timestamp,
       'CurrentAsOfFixedDemoObservation' AS data_freshness_indicator,
       'High' AS confidence_category,
       true AS advisory_only, true AS is_synthetic
FROM gold_dim_airport d
LEFT JOIN gold_airport_operational_health h ON d.airport_id = h.airport_id
LEFT JOIN baggage b ON d.airport_id = b.airport_id
LEFT JOIN staffing s ON d.airport_id = s.airport_id
LEFT JOIN retail r ON d.airport_id = r.airport_id
LEFT JOIN cx x ON d.airport_id = x.airport_id
""", 'gold_data_agent_enterprise_context')

In [ ]:
gold_contracts = [
    'gold_airline_route_performance','gold_baggage_performance','gold_workforce_coverage',
    'gold_retail_performance','gold_customer_experience','gold_turnaround_phase_performance',
    'gold_persona_scorecard','gold_data_agent_enterprise_context','gold_flight_operations_kpi',
    'gold_passenger_flow_kpi','gold_baggage_kpi','gold_workforce_kpi','gold_maintenance_kpi',
    'gold_energy_sustainability_kpi','gold_commercial_kpi','gold_incident_customer_kpi','gold_kpi_catalog']
for table_name in gold_contracts:
    frame = spark.table(table_name)
    assert frame.count() > 0 and frame.filter(~F.col('is_synthetic')).count() == 0

for table_name in [
    'gold_flight_operations_kpi','gold_passenger_flow_kpi','gold_baggage_kpi','gold_workforce_kpi',
    'gold_maintenance_kpi','gold_energy_sustainability_kpi','gold_commercial_kpi','gold_incident_customer_kpi']:
    assert spark.table(table_name).count() == spark.table('dim_airport').count(), table_name

assert spark.table('gold_airline_route_performance').filter(
    (F.col('on_time_departure_pct') < 0) | (F.col('on_time_departure_pct') > 100) |
    (F.col('load_factor_pct') < 0) | (F.col('load_factor_pct') > 100)).count() == 0
assert spark.table('gold_baggage_performance').filter((F.col('within_demo_sla_pct') < 0) | (F.col('within_demo_sla_pct') > 100)).count() == 0
assert spark.table('gold_workforce_coverage').filter((F.col('staffing_coverage_pct') < 0) | (F.col('staffing_coverage_pct') > 100)).count() == 0
assert spark.table('gold_customer_experience').filter(
    (F.col('satisfaction_score') < 1) | (F.col('satisfaction_score') > 5) |
    (F.col('nps_proxy') < -100) | (F.col('nps_proxy') > 100)).count() == 0

source_booking_revenue = spark.table('fact_booking').agg(F.sum('ticket_revenue_proxy')).first()[0]
gold_booking_revenue = spark.table('gold_airline_route_performance').agg(F.sum('ticket_revenue_proxy')).first()[0]
assert abs(source_booking_revenue - gold_booking_revenue) < 0.01
source_bags = spark.table('fact_baggage_journey').count()
gold_bags = spark.table('gold_baggage_performance').agg(F.sum('checked_bags')).first()[0]
assert source_bags == gold_bags
source_retail_net = spark.table('fact_retail_pos').select(F.sum(F.col('gross_sales_proxy') - F.col('refund_proxy')).alias('net')).first()['net']
gold_retail_net = spark.table('gold_retail_performance').agg(F.sum('net_revenue_proxy').alias('net')).first()['net']
assert abs(source_retail_net - gold_retail_net) < 0.10
assert spark.table('fact_turnaround_phase').count() == spark.table('fact_flight_turnaround_events').count() * 5
assert spark.table('gold_persona_scorecard').count() == spark.table('dim_airport').count() * 7

agent = spark.table('gold_data_agent_enterprise_context')
assert agent.count() == spark.table('dim_airport').count()
assert agent.filter(~F.col('advisory_only') | F.col('source_table_references').contains('bronze_') | F.col('source_table_references').contains('silver_')).count() == 0
for prohibited_phrase in ['dispatch ','command ','control ','automatically ','reroute ','assign staff']:
    assert agent.filter(F.lower('recommendation_text').contains(prohibited_phrase)).count() == 0
print('PASS: enterprise Gold star, KPI ranges, reconciliation, personas, provenance, and advisory-only safety')

In [ ]:
# Rotation and inventory Gold contracts.
for table_name in ['gold_aircraft_rotation_kpi','gold_retail_inventory_kpi']:
    frame=spark.table(table_name)
    assert frame.count()==spark.table('dim_airport').count()
    assert frame.filter(~F.col('is_synthetic')).count()==0
assert spark.table('gold_aircraft_rotation_kpi').filter(F.col('overlap_count')!=0).count()==0
assert spark.table('gold_retail_inventory_kpi').filter(~F.col('reorder_rate_pct').between(0,100)).count()==0
print('PASS: aircraft rotation and retail inventory Gold KPI contracts')